In [1]:
import pandas as pd

import random


In [2]:
df_art= pd.read_csv('art.csv')

In [3]:
def make_art_cite(row):
    artist = row['artist']
    title = row['title']
    year = row['year']
    return f'{artist}, *{title}*, {year}'

df_art['cite'] = df_art.apply(make_art_cite, axis=1)
df_art['link'] = '![](Images/' + df_art['slug'] + '.png)'

In [4]:
art_table = df_art.sort_values(by='year')[['link','cite']].to_markdown(index=False)
with open('markdown/art.md', 'w') as outfile:
    outfile.write(art_table)

In [5]:
art_dict = df_art.set_index('slug')['cite'].to_dict()

In [6]:
prefix = '''[&nbsp;]{.motto}

'''

In [7]:
suffix = '''
::: center
About [[Crisis *&* Opportunity]{.smallcaps}](about.html)
:::'''

In [8]:
forthcoming = '''
# Forthcoming
* Thompson, Anna J. "A Survey of Crime among Negroes in Philadelphia." *Opportunity* Vol 4, July-Sept.
* Johnson, Charles S.  and Horace M. Bond. "The Investigation of Racial Differences Prior to 1910." *The Journal of Negro Education*, Vol. 3, No. 3, (Jul., 1934), pp. 328-339. 
* Frazier, E. Franklin "The Status of the Negro in the American Social Order." *The Journal of Negro Education*, Vol. 4, No. 3, (Jul., 1935), pp. 293-307 
* Reid, Ira De A. "Negro Immigration to the United States."  *Social Forces*, Mar., 1938, Vol. 16, No. 3 (Mar., 1938), pp. 411-417
* Andrews, Norman P. "The Negro in Politics" *The Journal of Negro History* 1920 5:4, 420-436. 
'''

In [9]:
df = pd.read_csv('articles.csv')
len(df)

61

In [10]:
df = df.sort_values(by='Year')

In [11]:


def apply_template(row):
    
    title = row['title']
    article_url = row['article_url'] + '.html'
    art_url = row['artpng'] + '.png'
    art_credit = art_dict[row['artpng']]
    journal = row['Journal']
    year = row['Year']
    author = row['author']
    
    if len(title)<30:
        title = '&nbsp;<br>' + title
                          
    template = f'''
::: article
## [{title}](articles/{article_url})
### {author}
[![](Images/{art_url})](articles/{article_url} "{art_credit}") 
*{journal}*, {year}.
:::
'''
    
    return template

In [12]:
df['md'] = df.apply(apply_template, axis=1)

In [13]:
order = ['Racial Identity', 'White Racism and Racial Violence',  'Great Migration and Urban Sociology',
        'Labor and Economics','Gender',  'Health and Populations', 'Social Movements', 'Methods','Crime', 'Education', 'Family',]

article_mds = ''
for category in order:
    article_mds = article_mds + f"# {category}\n"
    sdf = df[df['Category'] == category]
    article_mds = article_mds + ''.join(sdf['md'].values)

In [14]:
md = prefix + article_mds + suffix

In [15]:
with open('markdown/index.md' ,'w') as outfile:
    outfile.write(md)

In [16]:
# ! ./build

In [17]:
# build

In [18]:

v = random.randint(100, 300)
for article_url in df['article_url'].values:
    cmd = f'pandoc -s --wrap=none -o docs/articles/{article_url}.html --template=templates/article_html.template markdown/{article_url}.md  --css="article_style.css?id={v}"'
    ! {cmd}
    cmd = f'pandoc -o word/{article_url}.docx docs/articles/{article_url}.html'
    #! {cmd}


In [19]:
about = f'pandoc -s   -o docs/about.html --template=templates/html.template markdown/about.md  --css="style.css?id={v}" --metadata title="About"'
index = f'pandoc -s   -o docs/index.html --template=templates/html.template markdown/index.md  --css="style.css?id={v}" --metadata title="C&O"'
art =   f'pandoc -s   -o docs/art.html --template=templates/html.template markdown/art.md  --css="style.css?id={v}" --metadata title="Art"'
! {about}
! {index}
! {art}


In [20]:
len(df)

61

In [21]:
import yaml

In [22]:
total_wc = 0

order = ['Racial Identity', 'White Racism and Racial Violence',  'Great Migration and Urban Sociology',
        'Labor and Economics','Gender',  'Health and Populations', 'Social Movements', 'Methods','Crime', 'Education', 'Family',]

md_string = '# Current Articles\n\n'

for category in order:
    md_string = md_string + f"\n## {category}\n\n"
    sdf = df[df['Category'] == category]
    for url in sdf['article_url'].values:

        fn = f'markdown/{url}.md'
        with open(fn,'r') as infile:
            md = infile.read()
        yaml_string = md.split('---\n')[1]
        meta = yaml.safe_load(yaml_string)

        author = meta['author'][0]['name']
        citation = meta['citation']
        title = meta['title']
        word_count = len(' '.join(md.split('---\n')[2:]).split(' '))
        total_wc = total_wc + word_count

        md_string = md_string +  f'* {author}. <br>"{title}." <br>{citation}<br> Word count: {word_count:,.0f}\n'
        
print(f'{total_wc:,.0f}')

261,415


In [23]:
with open('summary.md', 'w') as outfile:
    outfile.write(md_string)

In [24]:
!pandoc -o summary.docx summary.md

In [25]:
word_count

5508